# 12 — SBI Validation (Synthetic Expert Data)

Validates the three SBI representations (`pooled`, `moments`, `single`) on synthetic
BE/SC animals: does each net's held-out UM MSE recover the **generating model**, and
how well are the **parameters** recovered? Mirrors NB11 (grid search), one panel set
per representation.

Nets are trained by `train_sbi.py` and conditioned by `run_sbi.py` (both heavy → cluster);
this notebook loads the per-rep results and plots. Cross-rep / cross-method consensus is
NB 23, not here.

In [ ]:
%matplotlib inline
from shared_setup import *

from plotting.cv import plot_confusion, plot_recovery
from utils.cv_utils import load_cv_results
from scripts.config import results_dir, snpe_networks_dir, SBI_REPRESENTATIONS
from scripts.providers import load_animals
from scripts.train_sbi import train_one
from scripts.run_sbi import condition_cohort

REPS = list(SBI_REPRESENTATIONS)   # ['pooled', 'moments', 'single']

## 0. Helpers

`plot_confusion` / `plot_recovery` are draw-only single panels; these compose them per rep.

In [ ]:
def show_model_id(cv, title=''):
    """Confusion matrix + accuracy for one rep's loaded CVResults."""
    if cv.comparison is None or len(cv.comparison) == 0:
        print(f'{title}: no comparison rows'); return
    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    plot_confusion(cv.comparison, ax=ax)
    if title:
        ax.set_title(f'{title}\n{ax.get_title()}')
    plt.show()


def show_recovery(cv, title=''):
    """One true-vs-recovered scatter per recovered parameter."""
    if cv.recovery is None or len(cv.recovery) == 0:
        print(f'{title}: no recovery rows'); return
    params = list(cv.recovery['param'].unique())
    fig, axes = plt.subplots(1, len(params), figsize=(3.0 * len(params), 3.0))
    axes = np.atleast_1d(axes)
    for ax, p in zip(axes, params):
        plot_recovery(cv.recovery, p, ax=ax)
    if title:
        fig.suptitle(title, y=1.03)
    plt.tight_layout(); plt.show()

## 1. Quick (local) smoke — pipeline check

Trains the six nets at a tiny budget and conditions a small cohort, just to confirm the pipeline runs end to end. **Slow-ish and torch is required — not for real numbers.**

### 1a. Configuration

Generate the smoke cohort first (small), from a terminal:
```
python -m scripts.validation.generate_synthetic_cohort --cohort sbi_smoke --smoke-test
```

In [ ]:
SMOKE_COHORT = 'sbi_smoke'
SMOKE_RUN = 'smoke'
SMOKE_FT = 'update_matrix'
SMOKE_SIMS = 500          # tiny — checks the pipeline, not the numbers

records = load_animals('synthetic', cohort=SMOKE_COHORT)
print(len(records), 'animals:', [r.animal_id for r in records])

### 1b. Train the six nets (smoke budget)

Skip if the nets already exist in `snpe_networks/`.

In [ ]:
for rep in REPS:
    for model in ('BE', 'SC'):
        train_one(rep, model, n_simulations=SMOKE_SIMS)
print('trained 6 nets ->', snpe_networks_dir())

### 1c. Condition + plot per rep

In [ ]:
smoke_dir = results_dir('sbi', SMOKE_RUN, SMOKE_COHORT, SMOKE_FT)
for rep in REPS:
    for model in ('BE', 'SC'):
        condition_cohort(records, rep, model, smoke_dir / rep, SMOKE_FT, n_repeats=2)

for rep in REPS:
    cv = load_cv_results(smoke_dir / rep)
    show_model_id(cv, title=f'[smoke] SBI [{rep}] — model ID')
    show_recovery(cv, title=f'[smoke] SBI [{rep}] — recovery')

## 2. Full run (cluster results)

### 2a. Configuration

In [ ]:
from pathlib import Path

COHORT = 'static_uniform'      # synthetic cohort name (must exist on disk)
RUN = 'full'
FIT_TARGET = 'update_matrix'

folder_path = results_dir('sbi', RUN, COHORT, FIT_TARGET)
print('SBI results root:', folder_path)
for rep in REPS:
    d = folder_path / rep
    print(f'  {rep:8} {"exists" if d.exists() else "MISSING":7}: {d}')

### 2b. Model identification (per rep)

In [ ]:
for rep in REPS:
    cv = load_cv_results(folder_path / rep)
    show_model_id(cv, title=f'SBI [{rep}] — model identification')

### 2c. Parameter recovery (per rep)

In [ ]:
for rep in REPS:
    cv = load_cv_results(folder_path / rep)
    show_recovery(cv, title=f'SBI [{rep}] — parameter recovery')

### 2d. Representation comparison

Model-ID accuracy across the three reps (the headline: is `single` usable, and how far behind `pooled`?).

In [ ]:
rows = []
for rep in REPS:
    cv = load_cv_results(folder_path / rep)
    comp = cv.comparison
    if comp is None or len(comp) == 0:
        continue
    rows.append({'rep': rep, 'n_animals': len(comp),
                 'accuracy': float(comp['correct'].mean())})
rep_summary = pd.DataFrame(rows)
display(rep_summary)

if len(rep_summary):
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.bar(rep_summary['rep'], rep_summary['accuracy'])
    ax.axhline(0.5, ls='--', c='grey', lw=1)
    ax.set_ylim(0, 1); ax.set_ylabel('model-ID accuracy')
    ax.set_title('Representation comparison')
    plt.show()

## Pipeline

How these results are produced (training is heavy → cluster; conditioning is cheap):

1. **Cohort** — `python -m scripts.validation.generate_synthetic_cohort --cohort static_uniform`
2. **Train** (six nets, array 0–5) — `python -m scripts.train_sbi --task-id $SLURM_ARRAY_TASK_ID`
3. **Condition** (six rep×model, array 0–5) — `python -m scripts.run_sbi --source synthetic --cohort static_uniform --run full --fit-target update_matrix --task-id $SLURM_ARRAY_TASK_ID`

Cross-rep / cross-method consensus (GS + the three SBI reps) is NB 23.